## Customer Support Assistant

## Business Problem

## Build an AI assistant that helps customers answer questions related to order cancellation,   refunds, replacements, payments, delivery tracking, shipping, product issues, and account support.

## My dataset has-
''' instruction,response,category,intent,flags 

In [1]:
!pip install -q datasets

In [2]:
## Load Dataset
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

print(dataset)

README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [3]:
## Convert to a Pandas DataFrame
import pandas as pd

df = dataset["train"].to_pandas()

print(df.shape)
df.head()


(26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [4]:
## Save the CSV
df.to_csv("/kaggle/working/customer_support.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


## Generate non_instruction_data.txt (Stage 1)

In [5]:
responses = (
    df["response"]
    .dropna()
    .astype(str)
    .drop_duplicates()
)

paragraph = []

with open("/kaggle/working/non_instruction_data.txt", "w", encoding="utf-8") as f:

    for text in responses:
        text = text.replace("\n", " ").strip()

        if len(text) < 20:
            continue

        paragraph.append(text)

        # Write every 5 responses as one paragraph
        if len(paragraph) == 5:
            f.write(" ".join(paragraph))
            f.write("\n\n")
            paragraph = []

    if paragraph:
        f.write(" ".join(paragraph))

print("non_instruction_data.txt created successfully!")

non_instruction_data.txt created successfully!


## Verify the data

In [6]:
with open("/kaggle/working/non_instruction_data.txt", "r", encoding="utf-8") as f:
    print(f.read(2000))

I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you. I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority. I can sense that you're seeking assistance with canceling your purchase with the purchase number {{Order Number}}. I apologize for any inconvenience caused, and I'm here to guide you through the process.  To cancel your purchase, please follow these steps:  1. Log into your account: Visit our {{Online Company Portal Info}} and sign in using your credentials. 2. Locate your order: Once logged in, navigate to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section to find the purchas

In [7]:
import os

os.listdir("/kaggle/working")

['non_instruction_data.txt', 'customer_support.csv', '__notebook__.ipynb']

In [8]:
import json

count = 0

with open("/kaggle/working/instruction_dataset.jsonl","w") as f:

    for _, row in df.iterrows():

        if pd.isna(row["instruction"]) or pd.isna(row["response"]):
            continue

        item={
            "instruction":row["instruction"].strip(),
            "response":row["response"].strip()
        }

        f.write(json.dumps(item)+"\n")
        count+=1

print(count)

26872


In [9]:
df = dataset["train"].to_pandas()
display(df.head())
print(df.columns)

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='object')


## Generate instruction_dataset.jsonl

In [10]:
import json
import pandas as pd

count = 0

with open("/kaggle/working/instruction_dataset.jsonl", "w", encoding="utf-8") as f:

    for _, row in df.iterrows():

        # Skip missing values
        if pd.isna(row["instruction"]) or pd.isna(row["response"]):
            continue

        item = {
            "instruction": row["instruction"].strip(),
            "response": row["response"].strip()
        }

        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        count += 1

print(f"Total examples written: {count}")

Total examples written: 26872


In [11]:
import os

print(os.listdir("/kaggle/working"))

['non_instruction_data.txt', 'customer_support.csv', '__notebook__.ipynb', 'instruction_dataset.jsonl']


## Base Model Evaluation
 ## (Before fine-tuning, your assignment requires evaluating the base model on domain-specific questions. This provides a baseline for comparison.)

In [12]:
## Create the reports folder
import os

os.makedirs("/kaggle/working/reports", exist_ok=True)

## Create the Markdown file

In [13]:
questions = [
    "How can I cancel my order?",
    "How do I track my shipment?",
    "When will I receive my refund?",
    "How do I replace a damaged product?",
    "Why did my payment fail?",
    "Can I change my delivery address?",
    "How long does shipping take?",
    "Can I return a product after delivery?",
    "How do I contact customer support?",
    "What should I do if I receive the wrong item?"
]

with open("/kaggle/working/reports/base_model_evaluation.md", "w", encoding="utf-8") as f:

    f.write("# Base Model Evaluation\n\n")

    f.write("| Question | Base Model Answer | Problem |\n")
    f.write("|-----------|-------------------|---------|\n")

    for q in questions:
        f.write(f"| {q} | *(To be generated)* | *(To be evaluated)* |\n")

print("Template created.")

Template created.


In [14]:
import os

os.listdir("/kaggle/working/reports")
with open("/kaggle/working/reports/base_model_evaluation.md", "r", encoding="utf-8") as f:
    print(f.read())

# Base Model Evaluation

| Question | Base Model Answer | Problem |
|-----------|-------------------|---------|
| How can I cancel my order? | *(To be generated)* | *(To be evaluated)* |
| How do I track my shipment? | *(To be generated)* | *(To be evaluated)* |
| When will I receive my refund? | *(To be generated)* | *(To be evaluated)* |
| How do I replace a damaged product? | *(To be generated)* | *(To be evaluated)* |
| Why did my payment fail? | *(To be generated)* | *(To be evaluated)* |
| Can I change my delivery address? | *(To be generated)* | *(To be evaluated)* |
| How long does shipping take? | *(To be generated)* | *(To be evaluated)* |
| Can I return a product after delivery? | *(To be generated)* | *(To be evaluated)* |
| How do I contact customer support? | *(To be generated)* | *(To be evaluated)* |
| What should I do if I receive the wrong item? | *(To be generated)* | *(To be evaluated)* |



In [15]:
import os

print("Files in /kaggle/working:")
for file in os.listdir("/kaggle/working"):
    print(file)

Files in /kaggle/working:
reports
non_instruction_data.txt
customer_support.csv
__notebook__.ipynb
instruction_dataset.jsonl


## Verify before close this Notebook

In [16]:
## Check the reports folder
print("Files in /kaggle/working/reports:")

for file in os.listdir("/kaggle/working/reports"):
    print(file)

## Verify the CSV
import pandas as pd

df = pd.read_csv("/kaggle/working/customer_support.csv")

print(df.shape)
df.head()

## Verify non_instruction_data.txt

with open("/kaggle/working/non_instruction_data.txt", "r", encoding="utf-8") as f:
    print(f.read(1000))

## Verify instruction_dataset.jsonl

with open("/kaggle/working/instruction_dataset.jsonl", "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline())

Files in /kaggle/working/reports:
base_model_evaluation.md
(26872, 5)
I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you. I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority. I can sense that you're seeking assistance with canceling your purchase with the purchase number {{Order Number}}. I apologize for any inconvenience caused, and I'm here to guide you through the process.  To cancel your purchase, please follow these steps:  1. Log into your account: Visit our {{Online Company Portal Info}} and sign in using your credentials. 2. Locate your order: Once logged in, navigate to the '{{Online Order Intera

In [17]:
## Verify base_model_evaluation.md
with open("/kaggle/working/reports/base_model_evaluation.md", "r", encoding="utf-8") as f:
    print(f.read())

# Base Model Evaluation

| Question | Base Model Answer | Problem |
|-----------|-------------------|---------|
| How can I cancel my order? | *(To be generated)* | *(To be evaluated)* |
| How do I track my shipment? | *(To be generated)* | *(To be evaluated)* |
| When will I receive my refund? | *(To be generated)* | *(To be evaluated)* |
| How do I replace a damaged product? | *(To be generated)* | *(To be evaluated)* |
| Why did my payment fail? | *(To be generated)* | *(To be evaluated)* |
| Can I change my delivery address? | *(To be generated)* | *(To be evaluated)* |
| How long does shipping take? | *(To be generated)* | *(To be evaluated)* |
| Can I return a product after delivery? | *(To be generated)* | *(To be evaluated)* |
| How do I contact customer support? | *(To be generated)* | *(To be evaluated)* |
| What should I do if I receive the wrong item? | *(To be generated)* | *(To be evaluated)* |

